# Кейс 1 — парк вычислительных ускорителей OpenAI и Anthropic

**Дата оценки:** 13.09.2026  
**Вопрос:** сколько вычислительной мощности доступно OpenAI и Anthropic и как она распределяется между обучением/исследованиями и инференсом?

### Короткий ответ
- **OpenAI:** ≈ **2,7 млн H100e** (сценарный диапазон ≈2,5–3,0 млн)
- **Anthropic:** ≈ **2,0 млн H100e** (сценарный диапазон ≈1,9–2,2 млн)
- **Распределение:** центральная оценка ≈ **50% инференс / 50% обучение и исследования**, разумный диапазон 45–55%.

> H100e — H100-equivalent. Это единица нормализованной вычислительной мощности, а не количество физических GPU. Она нужна, потому что компании используют разные ускорители (NVIDIA, AWS Trainium, Google TPU).


## Логика
1. Берём системную оценку Epoch AI на конец 2025 года как базу.
2. Добавляем только мощности, введённые в эксплуатацию после базовой даты.
3. Для общих площадок считаем только долю, которую разумно отнести к конкретной компании.
4. Будущие/законтрактованные, но ещё не введённые мощности не включаем.
5. Точный текущий split инференс / обучение публично не раскрывается, поэтому показываем центральную оценку и диапазон.


**Уточнение верхнего сценария:** добавка 200 000 H100e для Anthropic — неподтверждённое допущение исходной модели, а не установленный ввод мощности. Она не входит в базовый итог. Источники и числа перенесены из исходных материалов; повторная внешняя проверка публикаций при сборке не проводилась.


In [1]:
"""
Кейс 1 — оценка вычислительной мощности OpenAI и Anthropic
Дата оценки: 13.09.2026

Цель:
1) привести неоднородные парки ускорителей к H100-эквивалентам (H100e);
2) обновить системную оценку Epoch AI на конец 2025 до сентября 2026;
3) показать сценарный диапазон;
4) отдельно задать диапазон распределения между инференсом и обучением/исследованиями.

Важно:
- это аналитическая оценка, а не официальное раскрытие компаний;
- базовый расчёт исключает будущую/законтрактованную мощность;
- верхний сценарий Anthropic содержит неподтверждённую добавку 200 000 H100e;
- проценты для общих площадок — допущения модели.
"""

AS_OF = "2026-09-13"

# -----------------------------
# OpenAI
# -----------------------------
OPENAI_BASE_2025 = 1_743_000  # H100e, Epoch AI, конец 2025

openai_sites = [
    # name, h100e_end_2025, h100e_current, attribution_low, attribution_base, attribution_high
    ("Stargate Abilene",      255_000, 509_000, 1.00, 1.00, 1.00),
    ("CoreWeave Denton",       65_000, 253_000, 0.70, 0.80, 1.00),
    ("Fairwater Atlanta",     388_000, 769_000, 0.50, 0.65, 1.00),
    ("Fairwater Wisconsin",         0, 446_000, 0.50, 0.65, 1.00),
]

def scenario_total(baseline, rows, scenario_index):
    total = baseline
    detail = []
    for row in rows:
        name, old, current, low, base, high = row
        share = (low, base, high)[scenario_index]
        increment = current - old
        attributed = increment * share
        total += attributed
        detail.append((name, increment, share, attributed))
    return total, detail

openai_low, openai_low_detail = scenario_total(OPENAI_BASE_2025, openai_sites, 0)
openai_base, openai_base_detail = scenario_total(OPENAI_BASE_2025, openai_sites, 1)
openai_high, openai_high_detail = scenario_total(OPENAI_BASE_2025, openai_sites, 2)

# -----------------------------
# Anthropic
# -----------------------------
ANTHROPIC_BASE_2025 = 1_190_000  # H100e, Epoch AI, конец 2025

# Rainier / New Carlisle: 471k -> 686k
rainier_increment = 686_000 - 471_000

# Colossus 1: 276k H100e, весь центр доступен Anthropic
colossus1_h100e = 276_000

# SEC: ~325k NVIDIA GPU across Colossus 1 + Colossus 2.
# Epoch: Colossus 1 = ~230k physical GPU, значит ~95k GPU относятся к Colossus 2.
anthropic_colossus2_gpu = 325_000 - 230_000

# Epoch: Colossus 2 ≈1.112M H100e на 440k B200/B300
colossus2_h100e_per_gpu = 1_112_000 / 440_000
colossus2_base_h100e = anthropic_colossus2_gpu * colossus2_h100e_per_gpu

# Lake Mariner: 59k H100e, Anthropic — вероятный пользователь
lake_mariner_h100e = 59_000

# Сценарии:
# low: Colossus2 85% от базовой оценки, Lake Mariner 50%
# base: Colossus2 100%, Lake Mariner 80%
# high: Colossus2 115%, Lake Mariner 100% + 200k H100e возможного доп. 2026 compute
anthropic_low = (
    ANTHROPIC_BASE_2025
    + rainier_increment
    + colossus1_h100e
    + colossus2_base_h100e * 0.85
    + lake_mariner_h100e * 0.50
)
anthropic_base = (
    ANTHROPIC_BASE_2025
    + rainier_increment
    + colossus1_h100e
    + colossus2_base_h100e
    + lake_mariner_h100e * 0.80
)
anthropic_high = (
    ANTHROPIC_BASE_2025
    + rainier_increment
    + colossus1_h100e
    + colossus2_base_h100e * 1.15
    + lake_mariner_h100e
    + 200_000
)

# -----------------------------
# Распределение нагрузки
# -----------------------------
# Публичного точного текущего split нет.
INFERENCE_LOW = 0.45
INFERENCE_BASE = 0.50
INFERENCE_HIGH = 0.55

def fmt_m(x):
    return f"{x/1_000_000:.2f} млн H100e"

print(f"Дата оценки: {AS_OF}")
print()
print("OpenAI:")
print("  low :", fmt_m(openai_low))
print("  base:", fmt_m(openai_base))
print("  high:", fmt_m(openai_high))
print()
print("Anthropic:")
print("  low :", fmt_m(anthropic_low))
print("  base:", fmt_m(anthropic_base))
print("  high:", fmt_m(anthropic_high))
print()
print("Центральная оценка распределения:")
print(f"  инференс: {INFERENCE_BASE:.0%}")
print(f"  обучение и исследования: {1-INFERENCE_BASE:.0%}")
print(f"  разумный диапазон инференса: {INFERENCE_LOW:.0%}–{INFERENCE_HIGH:.0%}")

# Числа, используемые на слайде:
print()
print("Округление для слайда:")
print(f"  OpenAI ≈ {openai_base/1_000_000:.1f} млн H100e")
print(f"  Anthropic ≈ {anthropic_base/1_000_000:.1f} млн H100e")


Дата оценки: 2026-09-13

OpenAI:
  low : 2.54 млн H100e
  base: 2.68 млн H100e
  high: 3.01 млн H100e

Anthropic:
  low : 1.91 млн H100e
  base: 1.97 млн H100e
  high: 2.22 млн H100e

Центральная оценка распределения:
  инференс: 50%
  обучение и исследования: 50%
  разумный диапазон инференса: 45%–55%

Округление для слайда:
  OpenAI ≈ 2.7 млн H100e
  Anthropic ≈ 2.0 млн H100e


## Основные источники

1. **Epoch AI — AI Chip Users**: методика H100e и системные оценки вычислительной мощности.  
   https://epoch.ai/data/ai-chip-users
2. **Epoch AI — AI Chip Users Explorer**: OpenAI 1,743 млн H100e и Anthropic 1,190 млн H100e на конец 2025; OpenAI в 2025 примерно поровну распределяла compute между R&D и inference.  
   https://epoch.ai/latest/introducing-the-ai-chip-users-explorer
3. **Epoch AI — Stargate Abilene**: текущая и историческая мощность, пользователь OpenAI.  
   https://epoch.ai/data/ai-data-centers/directory/openai-stargate-abilene
4. **Epoch AI — CoreWeave Denton**: рост мощности, OpenAI как вероятный пользователь.  
   https://epoch.ai/data/ai-data-centers/directory/coreweave-denton-tx
5. **Epoch AI — Microsoft Fairwater Atlanta / Wisconsin**: текущая мощность и использование OpenAI/Microsoft.  
   https://epoch.ai/data/ai-data-centers/directory/microsoft-fairwater-atlanta  
   https://epoch.ai/data/ai-data-centers/directory/microsoft-fairwater-wisconsin
6. **OpenAI — MRC supercomputer networking**: Abilene и Fairwater используются OpenAI для frontier training.  
   https://openai.com/index/mrc-supercomputer-networking/
7. **Epoch AI — Anthropic/Amazon New Carlisle**: Rainier / Trainium capacity.  
   https://epoch.ai/data/ai-data-centers/directory/anthropic-amazon-new-carlisle
8. **Anthropic — Amazon compute collaboration**: >1 млн Trainium2 для training и serving Claude.  
   https://www.anthropic.com/news/anthropic-amazon-compute
9. **Anthropic — SpaceX compute deal**: Colossus 1 и связь дополнительного compute с пользовательской нагрузкой Claude.  
   https://www.anthropic.com/news/higher-limits-spacex
10. **SEC / SpaceX filing**: ≈325 тыс. NVIDIA GPU для Anthropic across Colossus 1 + Colossus 2.  
    https://www.sec.gov/Archives/edgar/data/1181412/000162828026040874/spacexukfwp.htm
11. **Epoch AI — Colossus 1 / Colossus 2 / Lake Mariner**.  
    https://epoch.ai/data/ai-data-centers/directory/colossus-1  
    https://epoch.ai/data/ai-data-centers/directory/colossus-2  
    https://epoch.ai/data/ai-data-centers/directory/anthropic-lake-mariner

### Ограничения
- Общая мощность — аналитическая оценка, а не официальное раскрытие компаний.
- Проценты для общих площадок являются допущениями модели.
- 45–55% для инференса — сценарный диапазон, не статистический доверительный интервал.
- H100e сравнивает вычислительную мощность, но не гарантирует одинаковую производительность на конкретной модели/нагрузке.
